# DSCI_575 Amazon Product Query Assistant
Alan Liu, Zhihao Xie

## Download Data

In [11]:
import os
import requests
from pathlib import Path

In [12]:
data_folder = Path("../data/raw")
data_folder.mkdir(parents=True, exist_ok=True)

In [4]:
# Asked Gemini how to download data from link to jsonl.gz
def download_data(url, file_name):
    save_path = data_folder / file_name

    if save_path.exists():
        print(f"File Already exists at: {save_path}")
        return
    
    print(f"Downloading to: {save_path}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(save_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print("Download complete!")

In [6]:
# download review data
url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Sports_and_Outdoors.jsonl.gz"
file_name = "Sports_and_Outdoors.jsonl.gz"
download_data(url, file_name)

# download meta data
url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Sports_and_Outdoors.jsonl.gz"
file_name = "meta_Sports_and_Outdoors.jsonl.gz"
download_data(url, file_name)

File Already exists at: ..\data\raw\Sports_and_Outdoors.jsonl.gz
File Already exists at: ..\data\raw\meta_Sports_and_Outdoors.jsonl.gz


## Load Data
- an overview of the dataset (fields, size, example records)
selection and justification of fields for retrieval
description of text preprocessing decisions

### An overview of the dataset

In [29]:
def count_num_of_records(file_path):
    number_of_records = 0
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # Asked Gemini: how to print JSON in format
            product_key = data.get("asin")
            if product_key:
                number_of_records += 1
    return number_of_records

In [14]:
meta_path = data_folder / "meta_Sports_and_Outdoors.jsonl.gz"
# print(f"Number of products: {count_num_of_records(meta_path)}")

### Inspection of sample records:
Looks like the title of the product is in `meta_Sports_and_Outdoors.jsonl.gz`

In [7]:
import gzip
import json
from collections import Counter

### Examine Meta Data
It looks like the following fields are useful:
- title
- average_rating
- features (after concating all the elements into one string)
- description
- price
- categories
- parent_asin

In [15]:
# Gemini's approach to reading .jsonl.gz without unzipping
with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        # Asked Gemini: how to print JSON in format
        print(data.keys())
        print(json.dumps(data, indent=4))
        print(data['videos'])
        print(data['store'])
        print(data['categories'])
        print(data['details'])
        break

dict_keys(['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together'])
{
    "main_category": null,
    "title": "Sure-Grip Zombie Wheels Low 59mm 4 Pack",
    "average_rating": 4.5,
    "rating_number": 84,
    "features": [
        "Pre-packaged in sets of 4",
        "Low profile 59mm x 38mm",
        "89a w/purple hub, 92a w/black hub, 95a w/red hub, 98a w/green hub",
        "Made in the U.S.A.",
        "Anodized Aluminum Hub"
    ],
    "description": [
        "All Zombie wheels are made in the USA. Zombie wheels feature anodized aluminum hubs for maximum durability and precise feel while maintaining rock solid stability. This allows our unique urethane compounds to deliver all your power to the floor. Choose the Zombie combination that fits your skating style and surface. Zombie Aluminum Core \u2013 Designed in house and manufactured using state of the 

### Examine Review Data
Tried to find one review of the above product. Relevant columns:
- title
- text

In [28]:
# Examine review data
data_folder = Path("../data/raw")
review_path = data_folder / "Sports_and_Outdoors.jsonl.gz"
# Gemini's approach to reading .jsonl.gz without unzipping
with gzip.open(review_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        # Asked Gemini: how to print JSON in format
        if data.get("parent_asin") == 'B01HDXC8AG':
            print(json.dumps(data, indent=4))
            break

{
    "rating": 5.0,
    "title": "Excellent wheels",
    "text": "These replaced older wheels.  I mean from 80s old.  They are right height and solid.  The grip is a bit more than I want but that is no fault of wheels. I knew when I bought them they\u2019d be grippier but I didn\u2019t want to go less grippy.  I\u2019m very happy with performance.",
    "images": [],
    "asin": "B0157O33ES",
    "parent_asin": "B01HDXC8AG",
    "user_id": "AEHRHCKAPFO5RSX3VM73MZUXYJNA",
    "timestamp": 1520302057097,
    "helpful_vote": 4,
    "verified_purchase": true
}


### Retrieve the first 200 products for PoC

In [8]:
import pandas as pd

In [18]:
def remove_unwanted_keys(unwanted_keys, data):
    for key in unwanted_keys:
        data.pop(key, None)

product_count = 1
product_list = list()
parent_asin_set = set()
threshold = 200
unwanted_keys = ['main_category', 'rating_number', 'images', 'videos', 'store', 'details', 'bought_together', 'subtitle', 'author']
merged_keys = ['title', 'average_rating', 'features', 'description', 'price', 'categories']
# Gemini's approach to reading .jsonl.gz without unzipping
with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)

        # Remove unwanted keys
        remove_unwanted_keys(unwanted_keys, data)

        # Combine list into one text
        data['features'] = ' '.join(data['features'])
        data['description'] = ' '.join(data['description'])
        data['categories'] = ' '.join(data['categories'])

        # Merge all contents
        # Asked Gemini: How to merge multiple items in Python dict to one item and avoid copying the same string too many times?
        parts = [str(data.get(k, "")) for k in merged_keys]
        data["merged_content"] = " | ".join(parts)
        remove_unwanted_keys(merged_keys, data)

        # Add to product
        product_list.append(data)
        parent_asin_set.add(data['parent_asin'])

        # Check with threshold
        
        product_count += 1
        
        if product_count >= threshold:
            break
        elif product_count % 100 == 0:
            print(f"Processing Product #{product_count}")

products_df = pd.DataFrame(product_list)
products_df.head()

Processing Product #100


,parent_asin,merged_content
0,B01HDXC8AG,Sure-Grip Zombie Wheels Low 59mm 4 Pack | 4.5 ...
1,B07R5BQ4YD,USGI Wet Weather Bag (Fоur Paсk) | 4.2 | | We...
2,B003K8GZ7G,NHL San Jose Sharks Team Logo Post Earrings | ...
3,B08GC4GBWB,Bont Skates - Prostar Purple Suede Professiona...
4,B07BYV947H,Team Golf Alamaba Crimson Tide Embroidered Tow...


In [21]:
# Identify the corresponding review data

data_folder = Path("../data/raw")
review_path = data_folder / "Sports_and_Outdoors.jsonl.gz"
review_list = list()
unwanted_keys = ['rating', 'images', 'images', 'asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
merged_keys = ['title', 'text']
line_count = 0

with gzip.open(review_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        # Check if desired parent_asin
        if data.get("parent_asin") not in parent_asin_set:
            continue

        # Remove unwanted keys
        remove_unwanted_keys(unwanted_keys, data)

        # Merge all contents
        parts = [str(data.get(k, "")) for k in merged_keys]
        data["merged_content"] = " | ".join(parts)
        remove_unwanted_keys(merged_keys, data)

        # Add to product
        review_list.append(data)
        line_count += 1

In [25]:
reviews_df = pd.DataFrame(review_list)

# Asked Gemini: How to group by the same key in dataframe and concatenate the all the strings in one column?
reviews_df = reviews_df.groupby("parent_asin")["merged_content"].agg(" | ".join).reset_index()
print(f"Length of reviews_df: {len(reviews_df)}")

print("Top 5 lines:")
print(reviews_df.head())

Length of reviews_df: 199
Top 5 lines:
  parent_asin                                     merged_content
0  B000AOA6GO  Very pleased | Very pleased with the mirror. I...
1  B000CS4510  The Mora Light My Fire knife is a better optio...
2  B000F5XQII  I liked the fact that it fits | DO NOT WASH IT...
3  B0012NIMYA  Huge. | This is a huge magnet, good for a larg...
4  B001C3AHWU  great goggles! | We've tried many brands of go...


In [27]:
merged_df = pd.merge(products_df, reviews_df, on='parent_asin', how='inner')
merged_df["full_content"] = merged_df.pop("merged_content_x").astype(str) + " | " + merged_df.pop("merged_content_y").astype(str)
merged_df.head()

,parent_asin,full_content
0,B01HDXC8AG,Sure-Grip Zombie Wheels Low 59mm 4 Pack | 4.5 ...
1,B07R5BQ4YD,USGI Wet Weather Bag (Fоur Paсk) | 4.2 | | We...
2,B003K8GZ7G,NHL San Jose Sharks Team Logo Post Earrings | ...
3,B08GC4GBWB,Bont Skates - Prostar Purple Suede Professiona...
4,B07BYV947H,Team Golf Alamaba Crimson Tide Embroidered Tow...


In [29]:
merged_df.to_csv("../data/processed/merged.csv")